# Session 11: from a question to a first number

Most of today is thinking, not typing: choosing a question and cutting it
down to something you can finish. But one piece of code decides whether a
plan is real or only sounds good, and it is worth doing before you commit
to anything:

**the one-number test.** Can you get a rough version of the answer, in a
notebook, in about fifteen lines? If yes, the project is the right size. If
you cannot even say what the number would be, the question is not ready.

This notebook runs the test on the worked example from
`sessions/session-11/solutions/worked-example-scope.md`, which is scoped on
the music data. Your own project gets the same treatment in notebook 02.

In [ ]:
import os
from pathlib import Path

# Walk up until we find the folder with the data/ directory in it.
here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent

os.chdir(here)
print("working from:", Path.cwd())

## The worked example, in one paragraph

The repetitive task: *every January I copy last year's listening into a
spreadsheet and compare it with the year before, genre by genre, by hand.*

The question that came out of it, after scoping:

> **Which genres did I listen to more in 2025 than in 2024, and which
> less?**

One question, one dataset (`data/clean/plays.csv`), one output (a small
table of minutes per genre per year, with the change). Everything else went
on the "could" list.

## Step 1: the five-minute first look

Before any question gets answered, look at what you actually have. Not
what you think the file contains: what it contains. These four things,
every time, on any new dataset:

1. how many rows and columns
2. what type pandas decided each column is
3. how many values are missing, per column
4. the range of the dates, if there are any

It is worth having as a function, because you will run it on your own data
in ten minutes' time.

In [ ]:
import pandas as pd


def first_look(df, date_column=None):
    """Print the four things to check on any new dataset."""
    print(f"rows: {len(df):,}   columns: {len(df.columns)}")
    print()
    print("types and missing values per column:")
    summary = pd.DataFrame({"type": df.dtypes, "missing": df.isna().sum()})
    print(summary)
    if date_column is not None:
        print()
        print(f"{date_column}: {df[date_column].min()} to {df[date_column].max()}")


plays = pd.read_csv("data/clean/plays.csv")
first_look(plays, date_column="played_at")

Read it as a checklist rather than a printout:

* 2,183 rows, which matches what we know. Good.
* `played_at` is `object`, which means text. Fine for today, because we only
  need the year, and the first four characters are the year.
* nothing is missing. On your own data, expect this table to be the
  interesting part.
* two full calendar years, so comparing 2024 with 2025 is fair. If the data
  had stopped in October 2025, the question would be unfair before any code
  was written. **That is a scoping problem you can only see by looking.**

In [ ]:
assert len(plays) == 2183
assert plays["played_at"].min() == "2024-01-01"
assert plays["played_at"].max() == "2025-12-31"

## Step 2: the one number

The question needs minutes per genre per year. That is a `groupby` on two
columns, from session 8.

In [ ]:
plays["year"] = plays["played_at"].str[:4]

per_genre_year = plays.groupby(["genre", "year"])["minutes_played"].sum()
per_genre_year.head(6)

That is right but awkward to read: one long column with two labels on
each row. `unstack("year")` takes the `year` label and turns it into
columns, which is the table you would have built in the spreadsheet.

In [ ]:
table = per_genre_year.unstack("year").round(1)
table["change"] = (table["2025"] - table["2024"]).round(1)
table = table.sort_values("change")
table

## Step 3: believe it, or not

A number you cannot check is a number you are hoping about. The cheapest
check: the table must add up to the same total as the raw data. Nothing
should have been lost or counted twice on the way.

In [ ]:
total_raw = plays["minutes_played"].sum()
total_table = table["2024"].sum() + table["2025"].sum()

print(f"raw total:   {total_raw:,.1f}")
print(f"table total: {total_table:,.1f}")

assert round(total_raw, 1) == 8980.4
assert abs(total_raw - total_table) < 0.5     # the table was rounded, so allow a little

8,980.4 against 8,980.5. The table rounded sixteen numbers before they
were added up, so a difference of a tenth is rounding, not a lost play. A
difference of 200 would have been a bug.

Now the answer, written as a sentence, because a sentence is what you will
say in session 12.

In [ ]:
gained = table["change"].idxmax()
lost = table["change"].idxmin()

print(f"Most gained: {gained} ({table.loc[gained, 'change']:+.1f} minutes)")
print(f"Most lost:   {lost} ({table.loc[lost, 'change']:+.1f} minutes)")

assert gained == "Jazz" and table.loc["Jazz", "change"] == 54.0
assert lost == "Electronic" and table.loc["Electronic", "change"] == -106.1

`:+.1f` is the `:.1f` from session 1 with a `+`, so gains show as `+54.0`
instead of just `54.0`. Small, and it makes the sign impossible to miss.

So the one-number test passes, in about fifteen lines. **The project is
the right size.**

## Step 4: the decision hiding in the question

Look at the totals per year before going on.

In [ ]:
per_year = plays.groupby("year")["minutes_played"].sum().round(1)
print(per_year)
print(f"change: {per_year['2025'] - per_year['2024']:+.1f}")

Total listening fell by 212.8 minutes. So when a genre "lost minutes",
part of that is just "I listened less overall". A different reading of the
same question is: which genres took a **bigger share** of my listening?

In [ ]:
share = (per_genre_year.unstack("year") / per_year * 100).round(1)
share["change"] = (share["2025"] - share["2024"]).round(1)
share.sort_values("change")

In [ ]:
assert share["change"].idxmin() == "Folk"
assert share["change"].idxmax() == "Jazz"

Jazz wins either way. But the biggest loser changes: **Electronic** by
minutes, **Folk** by share.

Neither is wrong. They answer two slightly different questions, and the
question as written ("listened to more") is the minutes one. That is a
decision, so it goes on the worksheet under *decisions*, with the reason,
and the share version goes on the *could* list.

This is the most useful thing a first number does: it shows you the
decisions your question was quietly making, while changing your mind is
still cheap. Session 9 said every cleaning decision changes the answer.
So does every scoping decision.

## Step 5: a rough chart

The "should" item. Rough is fine today; the point is to prove it can be
drawn, not to make it pretty. One chart, one question, the answer in the
title, as in session 9.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
colours = ["tab:red" if c < 0 else "tab:green" for c in table["change"]]
ax.barh(table.index, table["change"], color=colours)
ax.axvline(0, color="black", linewidth=0.8)

for y, value in enumerate(table["change"]):
    ax.annotate(f"{value:+.1f}", (value, y), va="center",
                ha="left" if value >= 0 else "right",
                xytext=(4 if value >= 0 else -4, 0), textcoords="offset points")

ax.set_title(f"{gained} grew most from 2024 to 2025, {lost} shrank most")
ax.set_xlabel("change in minutes played, 2025 minus 2024")
ax.set_xlim(-140, 90)
fig.tight_layout()
plt.show()

## Step 6: what this tells us about the plan

The must item took fifteen lines and the should item took about fifteen
more. So the honest estimate for the project is not "the code", which is
nearly done. It is the parts around it:

| Item | Honest estimate | Why |
|---|---|---|
| must: the table, from `project/pipeline.py` | 1 hour | the code above, moved into functions |
| should: the chart, saved as a PNG | 1 hour | most of it is labels and checking it reads well |
| could: the share version as well | 30 minutes | the code exists; deciding how to show both is the work |
| could: month by month, as a line chart | 2 hours | a new grouping, and 24 points per genre is a crowded chart |
| presenting it: README, a clean run, ten minutes of story | 2 hours | always longer than you think |

Your own project will not be this tidy, because your data is not this
tidy. That is exactly why the first look and the first number happen
today, in the session, and not the night before session 12.

## Step 7: the same thing, shaped like the pipeline

`project/pipeline.py` from session 10 is load, clean, analyse, save. Here
is the worked example in that shape. Each function does one job and
**returns**; only the bottom cell writes or prints. It is the session 3
rule again: functions that calculate return, and only the outer layer
prints (or, here, writes a file).

In [ ]:
def load(path):
    """Read the plays CSV into a DataFrame."""
    return pd.read_csv(path)


def clean(df):
    """Keep the columns the question needs, and add a year column."""
    df = df[["played_at", "genre", "minutes_played"]].copy()
    df["year"] = df["played_at"].str[:4]
    return df


def analyse(df):
    """Return minutes per genre per year, with the change, smallest change first."""
    result = df.groupby(["genre", "year"])["minutes_played"].sum().unstack("year").round(1)
    result["change"] = (result["2025"] - result["2024"]).round(1)
    return result.sort_values("change")


def save(result, path):
    """Write the result table to a CSV, creating the folder if needed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(path)

In [ ]:
out = Path("output") / "session-11" / "genre_change.csv"

result = analyse(clean(load("data/clean/plays.csv")))
save(result, out)
print(f"wrote {out} ({len(result)} genres)")

assert result.equals(table)

The last line checks the pipeline version gives exactly the table we built
by hand above. When you move notebook code into `project/pipeline.py`, an
`assert` like that is how you know nothing changed on the way.

## Next

`02-first-run-exercise.ipynb`: the same steps on **your** data. First look,
first number, then the first end-to-end run of `project/pipeline.py`.